In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ============================================================================
# COMPLETE AUGMENTATION PIPELINE FOR PILL IMAGES WITH CSV LABELS
# Run this entire cell to augment your dataset
# ============================================================================
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import random
from PIL import Image
import os
from tqdm import tqdm
import pandas as pd

# ============================================================================
# 1. CUSTOM AUGMENTATION CLASSES
# ============================================================================
class BlisterReflection(A.ImageOnlyTransform):
    """
    Mô phỏng phản xạ nhựa blister / túi nylon
    """
    def __init__(self, *, intensity=(0.08, 0.25), always_apply=False, p=0.4):
        super().__init__(always_apply=always_apply, p=p)
        self.intensity = intensity

    def apply(self, img, **params):
        h, w = img.shape[:2]
        overlay = img.copy().astype(np.float32)

        num_lines = random.randint(2, 5)
        alpha = random.uniform(*self.intensity)

        for _ in range(num_lines):
            x1, y1 = random.randint(0, w), random.randint(0, h)
            x2, y2 = random.randint(0, w), random.randint(0, h)
            thickness = random.randint(1, 3)

            cv2.line(
                overlay,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                thickness
            )

        blended = cv2.addWeighted(
            overlay,
            alpha,
            img.astype(np.float32),
            1 - alpha,
            0
        )

        return np.clip(blended, 0, 255).astype(np.uint8)

class CenteredGlare(A.ImageOnlyTransform):
    """
    Thêm vùng lóa ở trung tâm viên thuốc
    """
    def __init__(self, intensity_range=(0.3, 0.7), size_range=(0.2, 0.5),
                 always_apply=False, p=0.5):
        super().__init__(always_apply=always_apply, p=p)
        self.intensity_range = intensity_range
        self.size_range = size_range

    def apply(self, img, **params):
        h, w = img.shape[:2]
        result = img.copy().astype(np.float32)

        # Lóa ở gần trung tâm
        center_x = w // 2 + random.randint(-w//6, w//6)
        center_y = h // 2 + random.randint(-h//6, h//6)

        # Size của vùng lóa
        size_factor = random.uniform(*self.size_range)
        radius = int(min(h, w) * size_factor)

        # Tạo gradient
        y, x = np.ogrid[:h, :w]
        distance = np.sqrt((x - center_x)**2 + (y - center_y)**2)
        mask = np.clip(1 - (distance / radius), 0, 1)

        # Gaussian blur
        mask = cv2.GaussianBlur(mask.astype(np.float32), (31, 31), 0)

        # Apply glare
        intensity = random.uniform(*self.intensity_range)
        glare = np.ones_like(result) * 255 * intensity

        mask = mask[:, :, np.newaxis]
        result = result * (1 - mask) + glare * mask

        return np.clip(result, 0, 255).astype(np.uint8)


class CenteredOcclusion(A.ImageOnlyTransform):
    """
    Che khuất một phần viên thuốc từ cạnh vào
    """
    def __init__(self, num_occlusions=(1, 2), size_range=(0.15, 0.35),
                 always_apply=False, p=0.5):
        super().__init__(always_apply=always_apply, p=p)
        self.num_occlusions = num_occlusions
        self.size_range = size_range

    def apply(self, img, **params):
        h, w = img.shape[:2]
        result = img.copy()

        num = random.randint(*self.num_occlusions)

        for _ in range(num):
            # Random position từ cạnh vào
            side = random.choice(['top', 'bottom', 'left', 'right', 'corner'])

            size_h = int(h * random.uniform(*self.size_range))
            size_w = int(w * random.uniform(*self.size_range))

            if side == 'top':
                x = random.randint(0, max(1, w - size_w))
                y = 0
            elif side == 'bottom':
                x = random.randint(0, max(1, w - size_w))
                y = h - size_h
            elif side == 'left':
                x = 0
                y = random.randint(0, max(1, h - size_h))
            elif side == 'right':
                x = w - size_w
                y = random.randint(0, max(1, h - size_h))
            else:  # corner
                x = random.choice([0, max(0, w - size_w)])
                y = random.choice([0, max(0, h - size_h)])

            # Random color
            color_type = random.choice(['dark', 'gray', 'skin'])
            if color_type == 'dark':
                color = tuple(random.randint(0, 60) for _ in range(3))
            elif color_type == 'gray':
                gray_val = random.randint(80, 140)
                color = (gray_val, gray_val, gray_val)
            else:  # skin tone
                color = (
                    random.randint(180, 220),
                    random.randint(140, 180),
                    random.randint(120, 160)
                )

            # Vẽ shape
            shape_type = random.choice(['rect', 'ellipse', 'irregular'])

            mask = np.zeros((h, w), dtype=np.float32)

            if shape_type == 'rect':
                cv2.rectangle(result, (x, y), (x + size_w, y + size_h), color, -1)
                cv2.rectangle(mask, (x, y), (x + size_w, y + size_h), 1, -1)
            elif shape_type == 'ellipse':
                center = (x + size_w // 2, y + size_h // 2)
                axes = (size_w // 2, size_h // 2)
                cv2.ellipse(result, center, axes, 0, 0, 360, color, -1)
                cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)
            else:  # irregular
                pts = np.array([
                    [x, y + random.randint(0, size_h//4)],
                    [x + size_w//3, y],
                    [x + 2*size_w//3, y + random.randint(0, size_h//4)],
                    [x + size_w, y + size_h//3],
                    [x + size_w, y + 2*size_h//3],
                    [x + 2*size_w//3, y + size_h],
                    [x + size_w//3, y + size_h],
                    [x, y + 2*size_h//3]
                ])
                cv2.fillPoly(result, [pts], color)
                cv2.fillPoly(mask, [pts], 1)

            # Blur edges
            mask = cv2.GaussianBlur(mask, (21, 21), 0)
            mask = mask[:, :, np.newaxis]

            result = (img * (1 - mask) + result * mask).astype(np.uint8)

        return result


class RealisticBlur(A.ImageOnlyTransform):
    """
    Blur thực tế: kết hợp motion blur và defocus
    """
    def __init__(self, *,always_apply=False, p=0.5):
        super().__init__(always_apply=always_apply, p=p)

    def apply(self, img, **params):
        blur_type = random.choice(['motion', 'defocus', 'both'])
        result = img.copy()

        if blur_type in ['motion', 'both']:
            # Motion blur
            size = random.choice([5, 7, 9])
            angle = random.randint(0, 180)

            kernel = np.zeros((size, size))
            kernel[size//2, :] = np.ones(size)
            kernel = kernel / size

            M = cv2.getRotationMatrix2D((size//2, size//2), angle, 1.0)
            kernel = cv2.warpAffine(kernel, M, (size, size))

            result = cv2.filter2D(result, -1, kernel)

        if blur_type in ['defocus', 'both']:
            # Defocus blur
            ksize = random.choice([3, 5, 7, 9])
            result = cv2.GaussianBlur(result, (ksize, ksize), 0)

        return result


# ============================================================================
# 2. AUGMENTATION PIPELINES
# ============================================================================
def get_background_robust_block(p=0.3):
    return A.OneOf([
        A.RandomResizedCrop(
            height=224,
            width=224,
            scale=(0.75, 1.0),
            ratio=(0.9, 1.1),
            p=1.0
        ),
        A.PadIfNeeded(
            min_height=224,
            min_width=224,
            border_mode=cv2.BORDER_CONSTANT,
            value=255,
            p=1.0
        ),
    ], p=p)

def get_light_augmentation():
    """
    Light augmentation - cho dataset đã tốt
    """
    return A.Compose([
        A.Rotate(limit=180, p=0.7, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),

        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=30,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            p=0.4
        ),

        # ✅ ADD LIGHT PERSPECTIVE
        A.Perspective(
            scale=(0.01, 0.03),
            keep_size=True,
            pad_mode=cv2.BORDER_CONSTANT,
            pad_val=0,
            p=0.5
        ),

        A.OneOf([
            A.MotionBlur(blur_limit=5, p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.3),

        A.RandomBrightnessContrast(
            brightness_limit=0.15,
            contrast_limit=0.15,
            p=0.5
        ),

        A.OneOf([
            A.HueSaturationValue(
                hue_shift_limit=10,
                sat_shift_limit=15,
                val_shift_limit=10,
                p=1.0
            ),
            A.RGBShift(
                r_shift_limit=10,
                g_shift_limit=10,
                b_shift_limit=10,
                p=1.0
            ),
        ], p=0.3),

        A.CoarseDropout(
            max_holes=2,
            max_height=40,
            max_width=40,
            min_holes=1,
            min_height=15,
            min_width=15,
            fill_value=0,
            p=0.3
        ),
    ])


def get_medium_augmentation():
    """
    Medium augmentation - cân bằng
    """
    return A.Compose([
        A.Rotate(limit=180, p=0.85, border_mode=cv2.BORDER_CONSTANT, value=255),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),

        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.15,
            rotate_limit=45,
            border_mode=cv2.BORDER_CONSTANT,
            value=255,
            p=0.5
        ),

        A.Perspective(scale=(0.02, 0.05), p=0.25),

        # BLUR
        A.OneOf([
            A.MotionBlur(blur_limit=7, p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MedianBlur(blur_limit=7, p=1.0),
            A.Defocus(radius=(3, 7), alias_blur=(0.1, 0.5), p=1.0),
        ], p=0.5),

        # NOISE
        A.OneOf([
            A.GaussNoise(var_limit=(5, 25), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
        ], p=0.03),

        # LIGHTING
        A.RandomBrightnessContrast(
            brightness_limit=0.25,
            contrast_limit=0.25,
            p=0.7
        ),

        A.RandomGamma(gamma_limit=(80, 120), p=0.5),

        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),

        A.RandomShadow(
            shadow_roi=(0, 0.5, 1, 1),
            num_shadows_lower=1,
            num_shadows_upper=2,
            p=0.3
        ),

        # COLOR
        A.OneOf([
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1.0),
            A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1.0),
        ], p=0.5),

        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),

        # OCCLUSION
        A.CoarseDropout(
            max_holes=3,
            max_height=45,
            max_width=45,
            min_holes=1,
            min_height=18,
            min_width=18,
            fill_value=255,
            p=0.4
        ),

        A.GridDropout(
            ratio=0.25,
            unit_size_min=10,
            unit_size_max=25,
            fill_value=255,
            p=0.3
        ),
        # Background robustness (OPTION 4)
        get_background_robust_block(p=0.3),

        # Reflection (OPTION 1)
        BlisterReflection(p=0.3),
        # QUALITY
        A.ImageCompression(quality_lower=60, quality_upper=100, p=0.3),
        A.Downscale(scale_min=0.5, scale_max=0.9, p=0.25),
    ])


def get_heavy_augmentation():
    """
    Heavy augmentation - cho model robust
    """
    return A.Compose([
        A.Rotate(limit=180, p=0.95, border_mode=cv2.BORDER_CONSTANT, value=255),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),

        A.ShiftScaleRotate(
            shift_limit=0.08,
            scale_limit=0.2,
            rotate_limit=60,
            border_mode=cv2.BORDER_CONSTANT,
            value=255,
            p=0.6
        ),

        A.Perspective(scale=(0.02, 0.08), p=0.3),

        # BLUR - HEAVY
        RealisticBlur(always_apply=False,p=0.6),

        A.OneOf([
            A.MotionBlur(blur_limit=9, p=1.0),
            A.GaussianBlur(blur_limit=(5, 11), p=1.0),
            A.MedianBlur(blur_limit=9, p=1.0),
        ], p=0.4),

        # NOISE - HEAVY
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.1), intensity=(0.2, 0.7), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.8, 1.2), p=1.0),
        ], p=0.1),

        # LIGHTING - HEAVY
        A.RandomBrightnessContrast(
            brightness_limit=0.35,
            contrast_limit=0.35,
            p=0.8
        ),

        A.RandomGamma(gamma_limit=(70, 130), p=0.6),

        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.4),

        A.RandomShadow(
            shadow_roi=(0, 0.5, 1, 1),
            num_shadows_lower=1,
            num_shadows_upper=3,
            p=0.4
        ),

        A.RandomSunFlare(
            flare_roi=(0, 0, 1, 0.5),
            angle_lower=0,
            angle_upper=1,
            num_flare_circles_lower=2,
            num_flare_circles_upper=5,
            src_radius=100,
            p=0.25
        ),

        A.RandomFog(
            fog_coef_lower=0.1,
            fog_coef_upper=0.3,
            alpha_coef=0.08,
            p=0.15
        ),

        # Custom glare
        CenteredGlare(intensity_range=(0.3, 0.8), size_range=(0.2, 0.5), p=0.4),

        # COLOR - HEAVY
        A.OneOf([
            A.HueSaturationValue(hue_shift_limit=25, sat_shift_limit=35, val_shift_limit=25, p=1.0),
            A.RGBShift(r_shift_limit=25, g_shift_limit=25, b_shift_limit=25, p=1.0),
            A.ChannelShuffle(p=1.0),
        ], p=0.6),

        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15, p=0.5),

        # OCCLUSION - HEAVY
        CenteredOcclusion(num_occlusions=(1, 3), size_range=(0.15, 0.4), p=0.5),

        A.CoarseDropout(
            max_holes=5,
            max_height=55,
            max_width=55,
            min_holes=1,
            min_height=18,
            min_width=18,
            fill_value=255,
            p=0.5
        ),

        A.GridDropout(
            ratio=0.3,
            unit_size_min=8,
            unit_size_max=30,
            fill_value=255,
            p=0.35
        ),

        # QUALITY - HEAVY
        A.ImageCompression(quality_lower=50, quality_upper=100, p=0.4),
        A.Downscale(scale_min=0.4, scale_max=0.9, p=0.3),
    ])


# ============================================================================
# 3. LOAD LABELS FROM CSV
# ============================================================================

def load_labels_from_csv(csv_path, image_name_col='image_name', label_col='label'):
    """
    Load labels từ CSV file

    Args:
        csv_path: Path to CSV file
        image_name_col: Tên cột chứa tên ảnh
        label_col: Tên cột chứa label

    Returns:
        dict: {image_name: label}
    """
    try:
        df = pd.read_csv(csv_path)

        # Check columns
        if image_name_col not in df.columns or label_col not in df.columns:
            print(f"❌ CSV must have columns: '{image_name_col}' and '{label_col}'")
            print(f"Available columns: {list(df.columns)}")
            return None

        # Create mapping
        label_dict = dict(zip(df[image_name_col], df[label_col]))

        print(f"✅ Loaded {len(label_dict)} labels from CSV")
        print(f"📊 Label distribution:")
        print(df[label_col].value_counts().sort_index())

        return label_dict

    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return None


# ============================================================================
# 4. MAIN AUGMENTATION FUNCTION WITH CSV LABELS
# ============================================================================

def augment_dataset_with_csv(
    input_dir,
    output_dir,
    csv_path,
    augmentations_per_image=5,
    augmentation_level='medium',
    save_original=True,
    image_extensions=('.jpg', '.jpeg', '.png', '.bmp'),
    image_name_col='image_name',
    label_col='label'
):
    """
    Augment dataset với labels từ CSV và tạo CSV mới cho augmented data

    Args:
        input_dir: Thư mục chứa ảnh gốc
        output_dir: Thư mục lưu ảnh đã augment
        csv_path: Path to CSV file chứa labels
        augmentations_per_image: Số ảnh augment tạo ra từ mỗi ảnh gốc
        augmentation_level: 'light', 'medium', 'heavy'
        save_original: Có lưu ảnh gốc không
        image_extensions: Các định dạng ảnh cần xử lý
        image_name_col: Tên cột chứa tên ảnh trong CSV
        label_col: Tên cột chứa label trong CSV
    """

    # Tạo output directory
    os.makedirs(output_dir, exist_ok=True)

    # Load labels từ CSV
    print("📋 Loading labels from CSV...")
    label_dict = load_labels_from_csv(csv_path, image_name_col, label_col)

    if label_dict is None:
        print("❌ Failed to load labels. Aborting.")
        return

    print(f"\n{'='*60}")

    # Chọn augmentation pipeline
    if augmentation_level == 'light':
        transform = get_light_augmentation()
        print("🔵 Using LIGHT augmentation")
    elif augmentation_level == 'medium':
        transform = get_medium_augmentation()
        print("🟡 Using MEDIUM augmentation")
    else:  # heavy
        transform = get_heavy_augmentation()
        print("🔴 Using HEAVY augmentation")

    # Lấy danh sách tất cả ảnh trong input_dir
    all_images = []
    for img_name in os.listdir(input_dir):
        if img_name.lower().endswith(image_extensions):
            img_path = os.path.join(input_dir, img_name)
            all_images.append((img_path, img_name))

    print(f"📸 Found {len(all_images)} images in input directory")
    print(f"🔄 Creating {augmentations_per_image} augmentations per image")
    print(f"💾 Total images to generate: {len(all_images) * (augmentations_per_image + (1 if save_original else 0))}")
    print(f"\n{'='*60}")

    # Statistics
    total_saved = 0
    failed = 0
    not_in_csv = 0

    # List để lưu augmented labels
    augmented_data = []

    # Process images
    for img_path, img_name in tqdm(all_images, desc="Augmenting images"):
        try:
            # Kiểm tra xem ảnh có trong CSV không
            if img_name not in label_dict:
                print(f"\n⚠️  Image not in CSV: {img_name}")
                not_in_csv += 1
                continue

            label = label_dict[img_name]

            # Đọc ảnh
            image = cv2.imread(img_path)
            if image is None:
                print(f"\n❌ Cannot read: {img_path}")
                failed += 1
                continue

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Lấy tên file gốc (không có extension)
            base_name = os.path.splitext(img_name)[0]

            # Lưu ảnh gốc
            if save_original:
                original_name = f"{base_name}_original.png"
                original_output = os.path.join(output_dir, original_name)
                Image.fromarray(image).save(original_output)
                total_saved += 1

                # Thêm vào augmented_data
                augmented_data.append({
                    'image_name': original_name,
                    'label': label,
                    'augmentation_type': 'original'
                })

            # Tạo augmented images
            for aug_idx in range(augmentations_per_image):
                # Apply augmentation
                augmented = transform(image=image)
                aug_image = augmented['image']

                # Lưu ảnh
                aug_name = f"{base_name}_aug{aug_idx+1}{augmentation_level}.png"
                aug_output = os.path.join(output_dir, aug_name)
                Image.fromarray(aug_image).save(aug_output)
                total_saved += 1

                # Thêm vào augmented_data
                augmented_data.append({
                    'image_name': aug_name,
                    'label': label,
                    # 'augmentation_type': f'augmented_{aug_idx+1}{augmentation_level}'
                })

        except Exception as e:
            print(f"\n❌ Error processing {img_path}: {e}")
            failed += 1

    # Lưu augmented labels vào CSV mới
    output_csv_path = os.path.join(
    output_dir,
    f'{augmentation_level}_augmented_train_label.csv'
)
    df_augmented = pd.DataFrame(augmented_data)
    df_augmented.to_csv(output_csv_path, index=False)

    # Summary
    print(f"\n{'='*60}")
    print(f"✅ AUGMENTATION COMPLETE!")
    print(f"{'='*60}")
    print(f"📊 Statistics:")
    print(f"   - Original images in folder: {len(all_images)}")
    print(f"   - Images not in CSV: {not_in_csv}")
    print(f"   - Images processed: {len(all_images) - not_in_csv - failed}")
    print(f"   - Augmented images saved: {total_saved}")
    print(f"   - Failed: {failed}")
    print(f"   - Success rate: {((len(all_images) - not_in_csv - failed) / len(all_images) * 100):.1f}%")
    print(f"\n📁 Output images directory: {output_dir}")
    print(f"📋 Output CSV file: {output_csv_path}")
    print(f"\n📊 Augmented label distribution:")
    print(df_augmented['label'].value_counts().sort_index())
    print(f"{'='*60}")

    return total_saved, failed, not_in_csv


# ============================================================================
# 5. VISUALIZE AUGMENTATIONS (OPTIONAL)
# ============================================================================

def visualize_augmentations(image_path, augmentation_level='medium', num_examples=9):
    """
    Visualize augmentation results
    """
    import matplotlib.pyplot as plt

    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Cannot read image: {image_path}")
        return

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Get transform
    if augmentation_level == 'light':
        transform = get_light_augmentation()
    elif augmentation_level == 'medium':
        transform = get_medium_augmentation()
    else:
        transform = get_heavy_augmentation()

    # Create figure
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.ravel()

    # Show original
    axes[0].imshow(image)
    axes[0].set_title('ORIGINAL', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Show augmented
    for idx in range(1, num_examples):
        augmented = transform(image=image)
        aug_image = augmented['image']

        axes[idx].imshow(aug_image)
        axes[idx].set_title(f'Augmented {idx}', fontsize=12)
        axes[idx].axis('off')

    plt.tight_layout()

    # Save
    output_path = f'augmentation_preview_{augmentation_level}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"✅ Preview saved to: {output_path}")
    plt.show()


# ============================================================================
# 6. RUN AUGMENTATION WITH CSV
# ============================================================================

if __name__ == "__main__":
    INPUT_DIR = "train_full"
    BASE_OUTPUT_DIR = "data"
    CSV_PATH = "train_label_1.csv"

    IMAGE_NAME_COL = 'image_name'
    LABEL_COL = 'label'
    SAVE_ORIGINAL = True

    augmentation_plan = [
        ("light", 25),
        ("medium",0),
        ("heavy", 0),
    ]

    print("🚀 Starting ordered augmentation pipeline...\n")

    for level, n_aug in augmentation_plan:
        print(f"\n{'='*70}")
        print(f"🔁 Running {level.upper()} augmentation ({n_aug} per image)")
        print(f"{'='*70}")

        output_dir = os.path.join(BASE_OUTPUT_DIR, level)

        augment_dataset_with_csv(
            input_dir=INPUT_DIR,
            output_dir=output_dir,
            csv_path=CSV_PATH,
            augmentations_per_image=n_aug,
            augmentation_level=level,
            save_original=SAVE_ORIGINAL,
            image_name_col=IMAGE_NAME_COL,
            label_col=LABEL_COL
        )

    print("\n🎉 ALL AUGMENTATIONS COMPLETED SUCCESSFULLY!")
    print("📂 Outputs:")
    print(f"   - {BASE_OUTPUT_DIR}/light")
    print(f"   - {BASE_OUTPUT_DIR}/medium")
    print(f"   - {BASE_OUTPUT_DIR}/heavy")

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\albumentations\__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


🚀 Starting ordered augmentation pipeline...


🔁 Running LIGHT augmentation (25 per image)
📋 Loading labels from CSV...
✅ Loaded 654 labels from CSV
📊 Label distribution:
label
0     35
1     36
2     33
3     32
4     38
5     38
6     35
7     35
8     32
9     34
10    62
11    64
12    60
13    64
14    56
Name: count, dtype: int64

🔵 Using LIGHT augmentation
📸 Found 654 images in input directory
🔄 Creating 25 augmentations per image
💾 Total images to generate: 17004



Augmenting images: 100%|██████████| 654/654 [02:27<00:00,  4.44it/s]



✅ AUGMENTATION COMPLETE!
📊 Statistics:
   - Original images in folder: 654
   - Images not in CSV: 0
   - Images processed: 654
   - Augmented images saved: 17004
   - Failed: 0
   - Success rate: 100.0%

📁 Output images directory: D:/intro2ai/data\light
📋 Output CSV file: D:/intro2ai/data\light\light_augmented_train_label.csv

📊 Augmented label distribution:
label
0      910
1      936
2      858
3      832
4      988
5      988
6      910
7      910
8      832
9      884
10    1612
11    1664
12    1560
13    1664
14    1456
Name: count, dtype: int64

🔁 Running MEDIUM augmentation (0 per image)
📋 Loading labels from CSV...
✅ Loaded 654 labels from CSV
📊 Label distribution:
label
0     35
1     36
2     33
3     32
4     38
5     38
6     35
7     35
8     32
9     34
10    62
11    64
12    60
13    64
14    56
Name: count, dtype: int64

🟡 Using MEDIUM augmentation
📸 Found 654 images in input directory
🔄 Creating 0 augmentations per image
💾 Total images to generate: 654



Augmenting images: 100%|██████████| 654/654 [00:04<00:00, 143.02it/s]



✅ AUGMENTATION COMPLETE!
📊 Statistics:
   - Original images in folder: 654
   - Images not in CSV: 0
   - Images processed: 654
   - Augmented images saved: 654
   - Failed: 0
   - Success rate: 100.0%

📁 Output images directory: D:/intro2ai/data\medium
📋 Output CSV file: D:/intro2ai/data\medium\medium_augmented_train_label.csv

📊 Augmented label distribution:
label
0     35
1     36
2     33
3     32
4     38
5     38
6     35
7     35
8     32
9     34
10    62
11    64
12    60
13    64
14    56
Name: count, dtype: int64

🔁 Running HEAVY augmentation (0 per image)
📋 Loading labels from CSV...
✅ Loaded 654 labels from CSV
📊 Label distribution:
label
0     35
1     36
2     33
3     32
4     38
5     38
6     35
7     35
8     32
9     34
10    62
11    64
12    60
13    64
14    56
Name: count, dtype: int64

🔴 Using HEAVY augmentation
📸 Found 654 images in input directory
🔄 Creating 0 augmentations per image
💾 Total images to generate: 654



Augmenting images: 100%|██████████| 654/654 [00:04<00:00, 141.95it/s]


✅ AUGMENTATION COMPLETE!
📊 Statistics:
   - Original images in folder: 654
   - Images not in CSV: 0
   - Images processed: 654
   - Augmented images saved: 654
   - Failed: 0
   - Success rate: 100.0%

📁 Output images directory: D:/intro2ai/data\heavy
📋 Output CSV file: D:/intro2ai/data\heavy\heavy_augmented_train_label.csv

📊 Augmented label distribution:
label
0     35
1     36
2     33
3     32
4     38
5     38
6     35
7     35
8     32
9     34
10    62
11    64
12    60
13    64
14    56
Name: count, dtype: int64

🎉 ALL AUGMENTATIONS COMPLETED SUCCESSFULLY!
📂 Outputs:
   - D:/intro2ai/data/light
   - D:/intro2ai/data/medium
   - D:/intro2ai/data/heavy


In [3]:
import os
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models
import random
import numpy as np

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def set_seed(seed: int = 42):
    """
    Set random seed for full reproducibility.
    """
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic behavior (IMPORTANT)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Optional (PyTorch >= 1.8)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

    # For hash-based ops
    os.environ["PYTHONHASHSEED"] = str(seed)

    print(f"[INFO] Random seed set to {seed}")


In [ ]:
train_data_path = "data/light"
train_label_path = "data/light/light_augmented_train_label.csv"
train_labels = pd.read_csv(train_label_path)
train_dict = train_labels.to_dict(orient="records")

In [6]:
class TrainImageDataset(Dataset):
    def __init__(self, data_dict, input_path, transform=None):
        self.data_dict = data_dict
        self.input_path = input_path
        self.transform = transform

    def __len__(self):
        return len(self.data_dict)

    def __getitem__(self, idx):
        img_name = self.data_dict[idx]['image_name']
        label = self.data_dict[idx]['label']

        img_path = os.path.join(self.input_path, img_name)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import torch

BATCH_SIZE = 64

# Dataset
train_dataset = TrainImageDataset(
    data_dict=train_dict,
    input_path=train_data_path,
    transform=transform
)

# --------------------------------------------------
# 1️⃣ Lấy label của từng sample
# (giả sử train_dict: {image_name: class_label})
# --------------------------------------------------
labels = [row["label"] for row in train_dict]
labels = np.array(labels)
print(labels)
# --------------------------------------------------
# 2️⃣ Tính số lượng mẫu mỗi class
# --------------------------------------------------
class_counts = np.bincount(labels)
num_classes = len(class_counts)

# --------------------------------------------------
# 3️⃣ Tính weight cho từng class (inverse frequency)
# --------------------------------------------------
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()  # optional normalize

# --------------------------------------------------
# 4️⃣ Gán weight cho từng sample
# --------------------------------------------------
sample_weights = class_weights[labels]
sample_weights = torch.DoubleTensor(sample_weights)

# --------------------------------------------------
# 5️⃣ Weighted sampler
# --------------------------------------------------
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# --------------------------------------------------
# 6️⃣ DataLoader (❌ shuffle)
# --------------------------------------------------

if True:
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler
    )

    print("✓ Using WeightedRandomSampler")
print(len(train_dataset))
x, y = next(iter(train_dataloader))
print(y)
print(torch.bincount(y))

[ 3  3  3 ... 11 11 11]
✓ Using WeightedRandomSampler
17004
tensor([13,  3,  9,  1, 11,  9,  0,  3,  3, 13,  6, 12,  3, 10,  3,  9,  5,  2,
         0, 12,  1,  8,  8,  3,  7, 11, 11,  2,  2,  9,  7,  5,  3, 12, 13, 10,
         6, 10, 11, 10, 13, 13, 13,  6,  7,  7,  9,  8, 10,  1, 12,  1,  8, 10,
         7,  4,  8,  4,  4,  8,  2, 12,  9,  0])
tensor([3, 4, 4, 7, 3, 2, 3, 5, 6, 6, 6, 4, 5, 6])


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
NUM_CLASSES = 15

cuda


In [10]:
# !!! Do not change anything of this cell !!!
from torch import Tensor
from typing import Type

class BasicBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        expansion: int = 1,
        downsample: nn.Module = None,
    ) -> None:
        super(BasicBlock, self).__init__()

        self.expansion = expansion
        self.downsample = downsample
        self.conv1_layer = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        self.batch_norm1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_layer = nn.Conv2d(
            out_channels,
            out_channels * self.expansion,
            kernel_size=3,
            padding=1,
            bias=False,
        )

        self.batch_norm2 = nn.BatchNorm2d(out_channels * self.expansion)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1_layer(x)
        out = self.batch_norm1(out)
        out = self.relu(out)

        out = self.conv2_layer(out)
        out = self.batch_norm2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class CNN(nn.Module):
    def __init__(
        self,
        block: Type[BasicBlock],
        img_channels: int = 3,
        num_classes: int = 10,
    ) -> None:
        super(CNN, self).__init__()
        layers = [2, 2, 2, 2]
        self.expansion = 1

        self.in_channels = 64

        self.conv_layer = nn.Conv2d(
            in_channels=img_channels,
            out_channels=self.in_channels,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        self.batch_norm = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool_layer = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer_1 = self._make_layer(block, 64, layers[0])
        self.layer_2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer_3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer_4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool_layer = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_layer = nn.Linear(512 * self.expansion, num_classes)

    def _make_layer(
        self, block: Type[BasicBlock], out_channels: int, blocks: int, stride: int = 1
    ) -> nn.Sequential:
        downsample = None
        if stride != 1:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels,
                    out_channels * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels * self.expansion),
            )
        layers = []
        layers.append(
            block(self.in_channels, out_channels, stride, self.expansion, downsample)
        )
        self.in_channels = out_channels * self.expansion

        for i in range(1, blocks):
            layers.append(
                block(self.in_channels, out_channels, expansion=self.expansion)
            )
        return nn.Sequential(*layers)

    def freeze_backbone(self):
        """Freeze all layers except the final FC layer"""
        for param in self.conv_layer.parameters():
            param.requires_grad = False
        for param in self.batch_norm.parameters():
            param.requires_grad = False
        for param in self.layer_1.parameters():
            param.requires_grad = False
        for param in self.layer_2.parameters():
            param.requires_grad = False
        for param in self.layer_3.parameters():
            param.requires_grad = False
        for param in self.layer_4.parameters():
            param.requires_grad = False
        # Keep fc_layer trainable

    def unfreeze_backbone(self):
        """Unfreeze all layers"""
        for param in self.parameters():
            param.requires_grad = True

    def forward(self, x: Tensor) -> Tensor:
        x = self.conv_layer(x)
        x = self.batch_norm(x)
        x = self.relu(x)
        x = self.maxpool_layer(x)

        x = self.layer_1(x)
        x = self.layer_2(x)
        x = self.layer_3(x)
        x = self.layer_4(x)

        x = self.avgpool_layer(x)
        x = torch.flatten(x, 1)
        x = self.fc_layer(x)

        return x

model = CNN(block=BasicBlock, num_classes=NUM_CLASSES).to(device)

# Freeze backbone for the first few epochs
FREEZE_EPOCHS = 5  # Adjust this number as needed
model.freeze_backbone()
print(f"Backbone frozen for the first {FREEZE_EPOCHS} epochs. Only FC layer will be trained.")

Backbone frozen for the first 5 epochs. Only FC layer will be trained.


In [11]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [12]:
"""
Training script with CutMix and MixUp augmentation integrated
"""

from timeit import default_timer as timer
from sklearn.metrics import f1_score
import torch
import torch.nn as nn
import numpy as np
from torchvision.transforms import v2
from tqdm import tqdm


def train_model_with_cutmix_mixup(
    model,
    train_dataloader,
    criterion,
    optimizer,
    device,
    num_classes,
    epochs=100,
    freeze_epochs=10,
    use_cutmix_mixup=True,
    cutmix_mixup_prob=0.5
):
    """
    Train model with CutMix and MixUp augmentation.

    Args:
        model: PyTorch model
        train_dataloader: Training data loader
        criterion: Loss function
        optimizer: Optimizer
        device: Device to train on
        num_classes: Number of classes
        epochs: Number of training epochs
        freeze_epochs: Number of epochs before unfreezing backbone
        use_cutmix_mixup: Whether to use CutMix/MixUp
        cutmix_mixup_prob: Probability of applying CutMix/MixUp

    Returns:
        dict: Training results
    """

    model_results = {"train_loss": [], "train_acc": [], "train_f1": []}
    start_timer = timer()

    # Setup CutMix and MixUp
    if use_cutmix_mixup:
        cutmix = v2.CutMix(num_classes=num_classes)
        mixup = v2.MixUp(num_classes=num_classes)
        cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])
        print(f"CutMix/MixUp enabled with probability: {cutmix_mixup_prob}")

    for epoch in tqdm(range(epochs), desc="Training"):
        model.train()

        # Unfreeze backbone after certain epochs
        if epoch == freeze_epochs:
            model.unfreeze_backbone()
            print(f"\nEpoch {epoch}: Unfreezing backbone. All layers are now trainable.")

        train_loss, train_acc = 0, 0
        all_preds = []
        all_labels = []

        for batch, (X, y) in enumerate(train_dataloader):
            X, y = X.to(device), y.to(device)

            # Apply CutMix or MixUp with probability
            if use_cutmix_mixup and np.random.rand() < cutmix_mixup_prob:
                X, y = cutmix_or_mixup(X, y)

            # Forward pass
            optimizer.zero_grad()
            y_pred = model(X)

            # Calculate loss
            loss = criterion(y_pred, y)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Get predictions
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

            # Handle mixed labels (from CutMix/MixUp)
            if y.dim() > 1:  # One-hot encoded labels
                y_true = torch.argmax(y, dim=1)
            else:  # Regular labels
                y_true = y

            # Store predictions and labels for F1 calculation
            all_preds.extend(y_pred_class.cpu().numpy())
            all_labels.extend(y_true.cpu().numpy())

            # Calculate batch accuracy
            train_acc += (y_pred_class == y_true).sum().item() / len(y_pred)
            train_loss += loss.item()

        # Calculate F1 score for the epoch
        if len(np.unique(all_labels)) > 1:  # Check if we have multiple classes
            epoch_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        else:
            epoch_f1 = 0.0

        # Calculate averages
        train_loss /= len(train_dataloader)
        train_acc /= len(train_dataloader)

        # Store results
        model_results["train_loss"].append(train_loss)
        model_results["train_acc"].append(train_acc)
        model_results["train_f1"].append(epoch_f1)

        # Save checkpoint
        torch.save(model.state_dict(), f"model_epoch_{epoch}.pth")

        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"\nEpoch: {epoch+1}/{epochs} | "
                  f"Loss: {train_loss:.4f} | "
                  f"Acc: {train_acc:.4f} | "
                  f"F1: {epoch_f1:.4f}")

    end_timer = timer()
    print(f"\nTotal training time: {end_timer - start_timer:.3f} seconds")

    return model_results


# =============================================================================
# EXAMPLE USAGE WITH YOUR ORIGINAL CODE STRUCTURE
# =============================================================================

def train_original_style(
    model,
    train_dataloader,
    criterion,
    optimizer,
    device,
    NUM_CLASSES,
    EPOCHS=100,
    FREEZE_EPOCHS=10
):
    """
    Training with CutMix/MixUp in your original code style.
    """

    model_results = {"train_loss": [], "train_acc": [], "train_f1": []}
    start_timer = timer()

    # Initialize CutMix and MixUp
    cutmix = v2.CutMix(num_classes=NUM_CLASSES)
    mixup = v2.MixUp(num_classes=NUM_CLASSES)
    cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

    for epoch in tqdm(range(EPOCHS)):
        model.train()

        if epoch == FREEZE_EPOCHS:
            model.unfreeze_backbone()
            print(f"Epoch {epoch}: Unfreezing backbone. All layers are now trainable.")

        train_loss, train_acc = 0, 0
        all_preds = []
        all_labels = []

        for batch, (X, y) in enumerate(train_dataloader):
            X, y = X.to(device), y.to(device)

            # ========================================
            # APPLY CUTMIX OR MIXUP HERE (50% chance)
            # ========================================
            if np.random.rand() < 0.5:  # 50% probability
                X, y = cutmix_or_mixup(X, y)

            optimizer.zero_grad()
            y_pred = model(X)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()

            # Get predictions
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

            # ========================================
            # HANDLE MIXED LABELS FROM CUTMIX/MIXUP
            # ========================================
            if y.dim() > 1:  # One-hot encoded (from CutMix/MixUp)
                y_true = torch.argmax(y, dim=1)
            else:  # Regular class indices
                y_true = y

            # Store predictions and labels for F1 calculation
            all_preds.extend(y_pred_class.cpu().numpy())
            all_labels.extend(y_true.cpu().numpy())

            # Calculate batch accuracy
            train_acc += (y_pred_class == y_true).sum().item() / len(y_pred)
            train_loss += loss.item()

        # Calculate F1 score for the epoch
        if len(np.unique(all_labels)) > 1:
            epoch_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        else:
            epoch_f1 = 0.0

        # Calculate averages
        train_loss /= len(train_dataloader)
        train_acc /= len(train_dataloader)

        # Store results
        model_results["train_loss"].append(train_loss)
        model_results["train_acc"].append(train_acc)
        model_results["train_f1"].append(epoch_f1)

        torch.save(model.state_dict(), f"model_epoch_{epoch}.pth")

        print(f"Epoch: {epoch+1} | Train loss: {train_loss:.4f}, "
              f"Train acc: {train_acc:.4f}, Train F1: {epoch_f1:.4f}")

    end_timer = timer()
    print(f"Total training time: {end_timer - start_timer:.3f} seconds")

    return model_results


# =============================================================================
# ADVANCED: SCHEDULE CUTMIX/MIXUP PROBABILITY
# =============================================================================

def get_cutmix_mixup_prob(epoch, total_epochs, strategy='constant'):
    """
    Get CutMix/MixUp probability based on training progress.

    Args:
        epoch: Current epoch
        total_epochs: Total number of epochs
        strategy: 'constant', 'linear_decay', 'warmup'

    Returns:
        float: Probability to apply CutMix/MixUp
    """
    if strategy == 'constant':
        return 0.3

    elif strategy == 'linear_decay':
        # Start at 0.8 and decay to 0.2
        return 0.75 - (0.6 * epoch / total_epochs)

    elif strategy == 'warmup':
        # No augmentation for first 10%, then 0.5
        if epoch < total_epochs * 0.1:
            return 0.0
        return 0.35

    return 0.35


def train_with_scheduled_augmentation(
    model,
    train_dataloader,
    criterion,
    optimizer,
    device,
    NUM_CLASSES,
    EPOCHS=100,
    FREEZE_EPOCHS=10,
    schedule_strategy='constant'
):
    """
    Training with scheduled CutMix/MixUp probability.
    """

    model_results = {"train_loss": [], "train_acc": [], "train_f1": []}
    start_timer = timer()

    cutmix = v2.CutMix(num_classes=NUM_CLASSES)
    mixup = v2.MixUp(num_classes=NUM_CLASSES)
    cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

    for epoch in tqdm(range(EPOCHS)):
        model.train()

        if epoch == FREEZE_EPOCHS:
            model.unfreeze_backbone()
            print(f"Epoch {epoch}: Unfreezing backbone.")

        # Get current probability
        current_prob = get_cutmix_mixup_prob(epoch, EPOCHS, schedule_strategy)

        train_loss, train_acc = 0, 0
        all_preds = []
        all_labels = []

        for batch, (X, y) in enumerate(train_dataloader):
            X, y = X.to(device), y.to(device)

            # Apply with scheduled probability
            if np.random.rand() < current_prob:
                X, y = cutmix_or_mixup(X, y)

            optimizer.zero_grad()
            y_pred = model(X)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()

            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

            if y.dim() > 1:
                y_true = torch.argmax(y, dim=1)
            else:
                y_true = y

            all_preds.extend(y_pred_class.cpu().numpy())
            all_labels.extend(y_true.cpu().numpy())

            train_acc += (y_pred_class == y_true).sum().item() / len(y_pred)
            train_loss += loss.item()

        if len(np.unique(all_labels)) > 1:
            epoch_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        else:
            epoch_f1 = 0.0

        train_loss /= len(train_dataloader)
        train_acc /= len(train_dataloader)

        model_results["train_loss"].append(train_loss)
        model_results["train_acc"].append(train_acc)
        model_results["train_f1"].append(epoch_f1)

        torch.save(model.state_dict(), f"model_epoch_{epoch}_cutmix.pth")

        print(f"Epoch: {epoch+1} | Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"F1: {epoch_f1:.4f} | CutMix/MixUp prob: {current_prob:.2f}")

    end_timer = timer()
    print(f"Total training time: {end_timer - start_timer:.3f} seconds")

    return model_results


# =============================================================================
# HOW TO USE IN YOUR NOTEBOOK
# =============================================================================
model_results = train_with_scheduled_augmentation(
    model=model,
    train_dataloader=train_dataloader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    NUM_CLASSES=NUM_CLASSES,
    EPOCHS=200,
    FREEZE_EPOCHS=5,
    schedule_strategy='linear_decay'  # or 'constant', 'linear_decay'
)
EPOCHS=200


  0%|          | 1/200 [02:40<8:52:14, 160.48s/it]

Epoch: 1 | Loss: 2.5671, Acc: 0.1786, F1: 0.1670 | CutMix/MixUp prob: 0.75


  1%|          | 2/200 [04:13<6:38:40, 120.81s/it]

Epoch: 2 | Loss: 2.4141, Acc: 0.2580, F1: 0.2491 | CutMix/MixUp prob: 0.75


  2%|▏         | 3/200 [05:22<5:18:18, 96.94s/it] 

...

Epoch: 198 | Loss: 0.1164, Acc: 0.9790, F1: 0.9790 | CutMix/MixUp prob: 0.16


100%|█████████▉| 199/200 [3:56:44<01:09, 69.91s/it]

Epoch: 199 | Loss: 0.1126, Acc: 0.9828, F1: 0.9828 | CutMix/MixUp prob: 0.16


100%|██████████| 200/200 [3:57:54<00:00, 71.37s/it]

Epoch: 200 | Loss: 0.1682, Acc: 0.9789, F1: 0.9788 | CutMix/MixUp prob: 0.15
Total training time: 14274.661 seconds


In [ ]:
test_data_path = "cropped_224"

In [14]:
class TestImageDataset(Dataset):
    def __init__(self, input_path, transform=None):
        self.input_path = input_path
        self.img_names = os.listdir(input_path)
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.input_path, self.img_names[idx])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, img_path

In [15]:
BATCH_SIZE = 1
test_dataset = TestImageDataset(input_path=test_data_path, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [16]:
# cv

In [17]:
# import pandas as pd

# Paths
# csv_path = "D:/intro2ai/prediction.csv"
# xlsx_path = "D:/intro2ai/prediction.xlsx"

# Read CSV
# df = pd.read_csv(csv_path)

# Write to Excel
# df.to_excel(xlsx_path, index=False)

# print("Conversion completed successfully!")


In [32]:
import torch
import os

import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2

mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

tta_transforms = [
    # identity
    A.Compose([
        #A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]),

    # horizontal flip
    A.Compose([
        A.HorizontalFlip(p=1),
        #A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]),

    # vertical flip
    A.Compose([
        A.VerticalFlip(p=1),
        #A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]),

    # rotation
    A.Compose([
        A.Rotate(limit=10, p=1),
        #A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]),

    # perspective
    A.Compose([
        A.Perspective(scale=(0.05, 0.15), p=1),
        #A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]),

    # blur
    A.Compose([
        A.GaussianBlur(blur_limit=(3, 5), p=1),
        #A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]),

    # A.Compose([
    #     A.MotionBlur(blur_limit=5, p=0.5),
    #     #A.Normalize(mean=mean, std=std),
    #     ToTensorV2()
    # ]),

    # coarse dropout
    A.Compose([
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=1),
        #A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]),
]


import torch
import os
import numpy as np

def infer(
    model,
    dataloader,
    tta_transforms,
    device
):
    image_names = []
    final_preds = []

    model.eval()
    with torch.no_grad():
        for images, paths in dataloader:
            # images: Tensor [B, C, H, W] → NumPy [B, H, W, C]
            images_np = images.permute(0, 2, 3, 1).cpu().numpy()

            all_votes = []

            for tta in tta_transforms:
                aug_images = []

                for img in images_np:
                    augmented = tta(image=img)["image"]
                    aug_images.append(augmented)

                aug_images = torch.stack(aug_images).to(device)

                outputs = model(aug_images)
                preds = torch.argmax(outputs, dim=1)
                all_votes.append(preds)

            votes = torch.stack(all_votes).permute(1, 0)
            batch_preds = torch.mode(votes, dim=1).values

            image_names.extend([os.path.basename(p) for p in paths])
            final_preds.extend(batch_preds.cpu().numpy())

    return image_names, final_preds

def infer_no_tta(
    model,
    dataloader,
    device
):
    image_names = []
    final_preds = []

    model.eval()
    with torch.no_grad():
        for images, paths in dataloader:
            images = images.to(device)   # [B, C, H, W]

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            image_names.extend([os.path.basename(p) for p in paths])
            final_preds.extend(preds.cpu().numpy())

    return image_names, final_preds

import pandas as pd
from sklearn.metrics import f1_score

# --------------------
# Load GT
# --------------------
gt_df = pd.read_excel("D:/intro2ai/prediction.xlsx")
gt_dict = dict(zip(gt_df["image_name"], gt_df["label"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = []

for epoch in range(80, EPOCHS):
    print(f"\nEvaluating epoch {epoch}")

    # --------------------
    # Load model
    # --------------------
    model = CNN(block=BasicBlock, num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(
        torch.load(f"model_epoch_{epoch}_cutmix.pth", map_location=device)
    )
    model.eval()

    # --------------------
    # Inference (TTA)
    # --------------------
    image_names, all_preds = infer(
        model=model,
        dataloader=test_dataloader,
        tta_transforms=tta_transforms,
        device=device,
    )

    # --------------------
    # Inference (NO TTA)
    # --------------------
    # image_names, all_preds = infer_no_tta(
    #     model=model,
    #     dataloader=test_dataloader,
    #     device=device,
    # )

    # --------------------
    # Match GT labels
    # --------------------
    all_labels = [gt_dict[name] for name in image_names]

    # --------------------
    # Macro F1
    # --------------------
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    results.append({
        "epoch": epoch,
        "macro_f1": macro_f1
    })

    print(f"Epoch {epoch}: Macro-F1 = {macro_f1:.6f}")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

best_row = results_df.loc[results_df["macro_f1"].idxmax()]

print("\nBest model:")
print(f"Epoch    : {int(best_row['epoch'])}")
print(f"Macro F1 : {best_row['macro_f1']:.6f}")



Evaluating epoch 80
Epoch 80: Macro-F1 = 0.870301

Evaluating epoch 81
Epoch 81: Macro-F1 = 0.703178

Evaluating epoch 82
Epoch 82: Macro-F1 = 0.820810

Evaluating epoch 83
Epoch 83: Macro-F1 = 0.877272

Evaluating epoch 84
Epoch 84: Macro-F1 = 0.827334

Evaluating epoch 85
Epoch 85: Macro-F1 = 0.786115

Evaluating epoch 86
Epoch 86: Macro-F1 = 0.763310

Evaluating epoch 87
Epoch 87: Macro-F1 = 0.852021

Evaluating epoch 88
Epoch 88: Macro-F1 = 0.863568

Evaluating epoch 89
Epoch 89: Macro-F1 = 0.852188

Evaluating epoch 90
Epoch 90: Macro-F1 = 0.800851

Evaluating epoch 91
Epoch 91: Macro-F1 = 0.820333

Evaluating epoch 92
Epoch 92: Macro-F1 = 0.866165

Evaluating epoch 93
Epoch 93: Macro-F1 = 0.851432

Evaluating epoch 94
Epoch 94: Macro-F1 = 0.877000

Evaluating epoch 95
Epoch 95: Macro-F1 = 0.756892

Evaluating epoch 96
Epoch 96: Macro-F1 = 0.857475

Evaluating epoch 97
Epoch 97: Macro-F1 = 0.827660

Evaluating epoch 98
Epoch 98: Macro-F1 = 0.833780

Evaluating epoch 99
Epoch 99: 

In [ ]:
image_names = []
all_preds = []
all_labels = []
model = CNN(block=BasicBlock, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(f"model_epoch_{int(best_row['epoch'])}_cutmix.pth"))
#model.load_state_dict(torch.load(f"model_epoch_{61}.pth"))
model.eval()

image_names, all_preds = infer(
        model=model,
        dataloader=test_dataloader,
        tta_transforms=tta_transforms,
        device=device,
    )

df = pd.DataFrame({"image_name": image_names, "label": all_preds})

df.to_csv("predictions.csv", index=False)
print("Conversion completed!")

Conversion completed!
